# Ninai Adapter Smoke Tests

Validates all components of `ninai.adapters` in one shot.
Run this after any SDK change as a quick sanity check.

Install: `pip install "ninai[all]"`

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
SDK_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'sdk', 'python'))
if SDK_PATH not in sys.path:
    sys.path.insert(0, SDK_PATH)

import langchain_core
import google.adk
import ninai
from ninai import NinaiClient, EnterpriseFeatureRequired
from ninai.adapters.langchain import NinaiChatMessageHistory, NinaiSearchTool, NinaiMemoryTool, get_ninai_toolkit
from ninai.adapters.adk import get_ninai_adk_tools, NinaiADKMemoryService

print(f'langchain_core : {langchain_core.__version__}')
print(f'google-adk     : {google.adk.__version__}')
print(f'ninai SDK      : {ninai.__version__}')
print()
print('All imports OK.')


langchain_core : 1.2.28
google-adk     : 1.29.0
ninai SDK      : 0.0.1b1

All imports OK.


## Shared mock client

In [2]:
from unittest.mock import MagicMock
from types import SimpleNamespace

_STORE: dict = {}
_CTR = [0]

def _mock_create(**kwargs):
    _CTR[0] += 1
    m = SimpleNamespace(
        id=str(_CTR[0]), content=kwargs.get('content', ''),
        title=kwargs.get('title', ''), tags=kwargs.get('tags', []),
    )
    _STORE[m.id] = m
    return m

def _mock_search(query, **kwargs):
    items = [
        SimpleNamespace(memory_id=m.id, content=m.content, score=0.9, title=m.title)
        for m in list(_STORE.values())[-5:]
    ]
    return SimpleNamespace(items=items[:3], total=len(items))

client = MagicMock()
client.memories.create.side_effect = _mock_create
client.memories.search.side_effect = _mock_search
print('Mock client ready')


Mock client ready


## LangChain adapter

In [3]:
from langchain_core.messages import HumanMessage, AIMessage

hist = NinaiChatMessageHistory('s1', client)
hist.add_messages([HumanMessage(content='hello'), AIMessage(content='hi')])
assert len(hist.messages) == 2, 'history length'

tool = NinaiSearchTool(ninai_client=client)
out = tool._run('hello')
assert isinstance(out, str), 'search returns str'

mem_tool = NinaiMemoryTool(ninai_client=client)
result = mem_tool._run('important decision', tags='eng,arch')
assert 'Stored' in result, 'store returns confirmation'

toolkit = get_ninai_toolkit(client)
assert len(toolkit) == 2, 'toolkit has 2 tools'

print('LangChain adapter smoke: PASS')


LangChain adapter smoke: PASS


## Google ADK adapter

In [4]:
tools = get_ninai_adk_tools(client)
assert len(tools) == 2, 'adk tools count'
assert tools[0].name == 'ninai_search_memory'
assert tools[1].name == 'ninai_store_memory'

store_fn = tools[1].func
search_fn = tools[0].func
res = store_fn('adk smoke test fact', tags='test')
assert res['status'] == 'stored'

hits = search_fn('smoke test')
assert isinstance(hits['results'], list)

svc = NinaiADKMemoryService(ninai_client=client)
svc.save_session_event('s2', {'role': 'user', 'content': 'test event'})
recalled = svc.search('test event')
assert isinstance(recalled, list)

print('ADK adapter smoke: PASS')


ADK adapter smoke: PASS


## EnterpriseFeatureRequired exception

In [5]:
try:
    raise EnterpriseFeatureRequired('needs license', feature='enterprise.scim')
except EnterpriseFeatureRequired as e:
    assert e.feature == 'enterprise.scim'
    assert 'sansten.com' in str(e)

print('EnterpriseFeatureRequired smoke: PASS')


EnterpriseFeatureRequired smoke: PASS


## SDK core

In [6]:
from ninai import (
    NinaiClient, GoalPlannerAgent, GoalLinkingAgent,
    MetaAgent, ToolInvoker, InMemoryEventSink,
)

assert issubclass(NinaiClient, object)
sink = InMemoryEventSink()
assert hasattr(sink, 'events')

print('Ninai SDK core smoke: PASS')


Ninai SDK core smoke: PASS


## Summary

In [7]:
print('=' * 42)
print('All adapter smoke tests PASSED')
print('  ninai.adapters.langchain : OK')
print('  ninai.adapters.adk       : OK')
print('  EnterpriseFeatureRequired: OK')
print('  Ninai SDK core           : OK')
print('=' * 42)


All adapter smoke tests PASSED
  ninai.adapters.langchain : OK
  ninai.adapters.adk       : OK
  EnterpriseFeatureRequired: OK
  Ninai SDK core           : OK
